## Cell 1 - Import our tools and existing modules

In [ ]:
import json
from dataclasses import asdict
from datetime import datetime, timezone
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from platform import python_version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from auger_reco.data import event_json
from auger_reco.physics import plane_front


## Cell 2 - Locate the event and identify our code


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# Walk upward until we find this project's source package.
while not (PROJECT_ROOT / "src" / "auger_reco").is_dir():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Start this notebook inside the Auger project.")

    PROJECT_ROOT = PROJECT_ROOT.parent

EVENT_ID = "081847956000"
EVENT_PATH = PROJECT_ROOT / "data" / "raw" / "events" / f"Auger_{EVENT_ID}.json"
MINIMUM_STATIONS = 6
WEIGHTING_MODES = ("timing", "uniform") # Primary baseline, then comparator.

module_paths ={
    "adapter": Path(event_json.__file__).resolve(),
    "fitter": Path(plane_front.__file__).resolve()
}

for name, path in module_paths.items():
    if not path.is_relative_to(PROJECT_ROOT / 'src'):
        raise RuntimeError(f"The name was imported from a different project: {path}")

module_hashes = {
    name: sha256(path.read_bytes()).hexdigest()
    for name, path in module_paths.items()
}

event_sha256 = sha256(EVENT_PATH.read_bytes()).hexdigest()

print(f"Project: {PROJECT_ROOT}")
print(f"Event file: {EVENT_PATH}")

## Cell 3 - Load the station measurements

In [ ]:
fit_input = event_json.load_auger_plane_front_input(
    EVENT_PATH,
    selection=event_json.StationSelection.OFFICIAL_SELECTED,
    minimum_stations=MINIMUM_STATIONS
)

if fit_input.canonical_event_id != EVENT_ID:
    raise ValueError("The file contains a different event than requested.")

print(f"Event: {fit_input.canonical_event_id}")
print(
    f"Selected / triggered: "
    f"{fit_input.number_of_stations} / "
    f"{fit_input.triggered_station_count}"
)

print(f"Selection policy: {fit_input.selection_policy.value}")
print(f"Adapter flags: {fit_input.quality_flags}")

station_table = pd.DataFrame({
    "station_id": fit_input.station_ids,
    "station_name": fit_input.station_names,
    "x_m": fit_input.station_positions_m[:, 0],
    "y_m": fit_input.station_positions_m[:, 1],
    "z_m": fit_input.station_positions_m[:, 2],
    "t_ns": fit_input.observed_times_ns,
    "dt_ns": fit_input.timing_uncertainties_ns
})

display(station_table.head())

## Cell 4 - Run the two plane-front reconstructions

In [ ]:
fit_options = plane_front.PlaneFrontFitOptions(minimum_stations=MINIMUM_STATIONS)

fits = {}
summary_rows = []

for weighting in WEIGHTING_MODES:
    result = plane_front.fit_plane_front(
        fit_input.station_positions_m,
        fit_input.observed_times_ns,
        fit_input.timing_uncertainties_ns,
        weighting=weighting,
        options=fit_options,
    )

    # Integration checks, not a requirement to agree with Auger's answer.

    np.testing.assert_allclose(
        np.linalg.norm(result.sky_direction),
        1.0,
        rtol=0.0,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        result.propagation_direction,
        -result.sky_direction,
        rtol=0.0,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        result.residuals_ns,
        fit_input.observed_times_ns - result.predicted_times_ns,
        rtol=0.0,
        atol=1e-8,
    )

    fits[weighting] = result

    summary_rows.append({
        "weighting": weighting,
        "zenith_deg": result.zenith_deg,
        "azimuth_deg": result.azimuth_deg,
        "sky_x": result.sky_direction[0],
        "sky_y": result.sky_direction[1],
        "sky_z": result.sky_direction[2],
        "origin_time_ns": result.origin_time_ns,
        "chi_square": result.chi_square,
        "degrees_of_freedom": result.degrees_of_freedom,
        "reduced_chi_square": result.reduced_chi_square,
        "rms_residual_ns": result.rms_residual_ns,
        "maximum_absolute_residual_ns": result.maximum_absolute_residual_ns,
        "quality_flags": "; ".join(result.quality_flags),
    })

if set(fits) != set(WEIGHTING_MODES) or len(summary_rows) != len(WEIGHTING_MODES):
    raise RuntimeError(f"Expected both fit results; available keys: {list(fits)}")

prediction_summary = pd.DataFrame(summary_rows)

display(prediction_summary.loc[:, [
    "weighting",
    "zenith_deg",
    "azimuth_deg",
    "reduced_chi_square",
    "rms_residual_ns",
    "quality_flags",
]])

## Cell 5 - Save predictions and reproducibility information

In [ ]:
# Confirm that our recorded file identities still describe this run.
if sha256(EVENT_PATH.read_bytes()).hexdigest() != event_sha256:
    raise RuntimeError("The event file changed. Restart the kernel and run all cells.")

for name, path in module_paths.items():
    if sha256(path.read_bytes()).hexdigest() != module_hashes[name]:
        raise RuntimeError(
            f"The {name} source changed. Restart the kernel and run all cells."
        )

run_time = datetime.now(timezone.utc)
run_tag = run_time.strftime("%Y%m%dT%H%M%S_%fZ")

RUN_DIR = (
    PROJECT_ROOT
    / "reports"
    / "results"
    / f"event_{EVENT_ID}_run_{run_tag}"
)

RUN_DIR.mkdir(parents=True, exist_ok=False)

run_metadata = {
    "event_id": EVENT_ID,
    "event_sha256": event_sha256,
    "embedded_release": fit_input.embedded_release,
    "format_version": fit_input.format_version,
    "created_utc": run_time.isoformat(),
    "selected_stations": fit_input.number_of_stations,
    "triggered_stations": fit_input.triggered_station_count,
    "selection_policy": fit_input.selection_policy.value,
    "primary_method": "timing",
    "comparison_method": "uniform",
    "fit_options": asdict(fit_options),
    "source_sha256": module_hashes,
    "python_version": python_version(),
    "package_versions": {
        name: version(name)
        for name in ("numpy", "scipy", "pandas", "matplotlib")
    },
    "evaluation_role": "unblinded single-event integration example",
}

prediction_summary.to_csv(
    RUN_DIR / "predictions_before_reference.csv",
    index=False,
)

(RUN_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

# Keep optimizer diagnostics for both methods, including rejected attempts.
for weighting, result in fits.items():
    attempts = pd.DataFrame([
        asdict(attempt)
        for attempt in result.optimizer_attempts
    ])

    attempts.to_csv(
        RUN_DIR / f"{weighting}_optimizer_attempts.csv",
        index=False,
    )

    print(
        f"{weighting}: geometry rank={result.full_geometry_rank}, "
        f"Jacobian rank={result.final_jacobian_rank}, "
        f"surface condition={result.surface_condition_number:.3g}, "
        f"flags={result.quality_flags}"
    )

print(f"Run saved in: {RUN_DIR}")


## Cell 6 - Inspect the timing residuals

In [ ]:
primary_fit = fits["timing"]

diagnostics = station_table.copy()
diagnostics["predicted_time_ns"] = primary_fit.predicted_times_ns
diagnostics["residual_ns"] = primary_fit.residuals_ns
diagnostics["standardized_residual"] = primary_fit.standardized_residuals

diagnostics.to_csv(
    RUN_DIR / "weighted_station_diagnostics.csv",
    index=False,
)

# Subtract the SAME offset from observed and predicted times.
time_zero = fit_input.observed_times_ns.min()
observed_relative = fit_input.observed_times_ns - time_zero
predicted_relative = primary_fit.predicted_times_ns - time_zero

fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 5),
    layout="constrained",
)

axes[0].scatter(
    observed_relative,
    predicted_relative,
    s=35,
)

limits = [
    min(observed_relative.min(), predicted_relative.min()),
    max(observed_relative.max(), predicted_relative.max()),
]

axes[0].plot(
    limits,
    limits,
    color="black",
    linestyle="--",
)

axes[0].set_xlabel("Observed relative time (ns)")
axes[0].set_ylabel("Predicted relative time (ns)")
axes[0].set_title("Timing-weighted plane fit")

axes[1].errorbar(
    observed_relative,
    primary_fit.residuals_ns,
    yerr=fit_input.timing_uncertainties_ns,
    fmt="o",
    capsize=3,
)

axes[1].axhline(
    0.0,
    color="black",
    linestyle="--",
)

axes[1].set_xlabel("Observed relative time (ns)")
axes[1].set_ylabel("Observed − predicted time (ns)")
axes[1].set_title("Residuals; bars show input dt")

# Symmetric colour limits put zero at the centre of the colour scale.
colour_limit = max(
    float(np.max(np.abs(primary_fit.residuals_ns))),
    1e-12,
)

footprint = axes[2].scatter(
    fit_input.station_positions_m[:, 0],
    fit_input.station_positions_m[:, 1],
    c=primary_fit.residuals_ns,
    cmap="coolwarm",
    vmin=-colour_limit,
    vmax=colour_limit,
    s=55,
    edgecolors="black",
)

axes[2].set_xlabel("Station x (m)")
axes[2].set_ylabel("Station y (m)")
axes[2].set_title("Selected-station residual map")
axes[2].set_aspect("equal", adjustable="box")

fig.colorbar(
    footprint,
    ax=axes[2],
    label="Timing residual (ns)",
)

for ax in axes:
    ax.grid(alpha=0.25)

fig.savefig(
    RUN_DIR / "weighted_timing_diagnostics.png",
    dpi=200,
)

plt.show()
display(diagnostics.head())

## Cell 7 - Compare with Auger’s published direction

In [ ]:
if sha256(EVENT_PATH.read_bytes()).hexdigest() != event_sha256:
    raise RuntimeError(
        "The event file changed. Restart the kernel and run all cells."
    )

reference = event_json.load_auger_direction_reference(EVENT_PATH)

if (
    reference.event_id != fit_input.event_id
    or reference.embedded_release != fit_input.embedded_release
    or reference.format_version != fit_input.format_version
):
    raise ValueError(
        "The reference does not match the fitted event metadata."
    )

reference_sky = plane_front.sky_direction_from_angles(
    np.deg2rad(reference.zenith_deg),
    np.deg2rad(reference.azimuth_deg),
)

comparison = prediction_summary.copy()

comparison["auger_zenith_deg"] = reference.zenith_deg
comparison["auger_azimuth_deg"] = reference.azimuth_deg

comparison["separation_from_auger_deg"] = [
    plane_front.angular_separation_deg(
        fits[method].sky_direction,
        reference_sky,
    )
    for method in comparison["weighting"]
]

comparison.to_csv(
    RUN_DIR / "comparison_with_auger.csv",
    index=False,
)

display(comparison.loc[:, [
    "weighting",
    "zenith_deg",
    "azimuth_deg",
    "auger_zenith_deg",
    "auger_azimuth_deg",
    "separation_from_auger_deg",
]])

print(f"All run artifacts: {RUN_DIR}")